# Data API Collector — Databricks Quick Start

Verify connectivity to all services and run a quick end-to-end test.

**For deep-dive examples by capability, see:**
- [`examples/kafka_streaming.ipynb`](examples/kafka_streaming.ipynb) — 9 Kafka streaming use cases (core + SLED)
- [`examples/neo4j_graph.ipynb`](examples/neo4j_graph.ipynb) — Populate and query SLED graph data
- [`examples/postgres_jdbc.ipynb`](examples/postgres_jdbc.ipynb) — Populate and read SLED relational tables

Each notebook includes its own configuration cell — update HOST and secret scope in each notebook you use.

In [ ]:
%pip install neo4j
dbutils.library.restartPython()

In [ ]:
# ---------------------------------------------------------------------------
# Configuration — UPDATE THESE for your environment
# ---------------------------------------------------------------------------
HOST = "your-hostname.com"

# Secrets — loaded from Databricks secret scope
# Set up with:
#   databricks secrets create-scope data-api
#   databricks secrets put-secret data-api api-key --string-value "YOUR_SECRET_KEY"
#   databricks secrets put-secret data-api kafka-user --string-value "YOUR_KAFKA_USER"
#   databricks secrets put-secret data-api kafka-password --string-value "YOUR_KAFKA_PASSWORD"
#   databricks secrets put-secret data-api neo4j-password --string-value "YOUR_NEO4J_PASSWORD"
#   databricks secrets put-secret data-api postgres-password --string-value "YOUR_POSTGRES_PASSWORD"

API_KEY           = dbutils.secrets.get(scope="data-api", key="api-key")
KAFKA_USER        = dbutils.secrets.get(scope="data-api", key="kafka-user")
KAFKA_PASSWORD    = dbutils.secrets.get(scope="data-api", key="kafka-password")
NEO4J_USER        = "neo4j"
NEO4J_PASSWORD    = dbutils.secrets.get(scope="data-api", key="neo4j-password")
POSTGRES_USER     = "postgres"
POSTGRES_PASSWORD = dbutils.secrets.get(scope="data-api", key="postgres-password")

# Derived URLs
API_URL         = f"https://{HOST}/api/v1"
KAFKA_BROKER    = f"{HOST}:9093"
NEO4J_URI       = f"bolt+s://{HOST}:7688"
POSTGRES_JDBC   = f"jdbc:postgresql://{HOST}:15433/datacollector?ssl=true&sslmode=require"
HEADERS         = {"X-Api-Key": API_KEY}

# Checkpoint base path — update to your catalog/schema
CHECKPOINT_BASE = "/Volumes/your_catalog/your_schema/checkpoints"

# SLED use cases
SLED_USE_CASES = [
    "student_enrollment", "grant_budget", "citizen_services",
    "k12_early_warning", "procurement", "case_management",
]

# Kafka connection options (reusable)
KAFKA_OPTIONS = {
    "kafka.bootstrap.servers": KAFKA_BROKER,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="{KAFKA_USER}" '
        f'password="{KAFKA_PASSWORD}";'
    ),
    "startingOffsets": "earliest",
}

# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------
import requests
import builtins
from pyspark.sql.types import *
from pyspark.sql.functions import *


def read_kafka_stream(topic, schema):
    """Read a Kafka topic as a parsed Structured Streaming DataFrame."""
    raw = spark.readStream.format("kafka")
    for k, v in KAFKA_OPTIONS.items():
        raw = raw.option(k, v)
    raw = raw.option("subscribe", topic).load()
    return (
        raw
        .select(from_json(col("value").cast("string"), schema).alias("data"))
        .select("data.*")
    )


def read_pg_table(table_name):
    """Read a table from PostgreSQL via JDBC."""
    return (
        spark.read
        .format("jdbc")
        .option("url", POSTGRES_JDBC)
        .option("dbtable", table_name)
        .option("user", POSTGRES_USER)
        .option("password", POSTGRES_PASSWORD)
        .option("driver", "org.postgresql.Driver")
        .load()
    )


print(f"Config loaded — API: {API_URL}, Kafka: {KAFKA_BROKER}")


In [ ]:
import neo4j

---
## 1. Service Health Checks

In [ ]:
checks = {
    "PostgreSQL ORM":  "/data-sources/orm",
    "PostgreSQL SQL":  "/data-sources/raw/sql",
    "Neo4j Health":    "/data-sources/neo4j/health",
    "Neo4j Version":   "/data-sources/neo4j/version",
    "Redis":           "/redis/test",
    "Kafka Events":    "/kafka/events?limit=1",
}

for name, path in checks.items():
    try:
        r = requests.get(f"{API_URL}{path}", headers=HEADERS, timeout=10)
        status = r.json().get("status", r.status_code)
        print(f"  {name}: {status}")
    except Exception as e:
        print(f"  {name}: FAILED — {e}")

---
## 2. Quick Kafka Test

In [ ]:
resp = requests.post(f"{API_URL}/kafka/generators/start", headers=HEADERS, json={
    "use_case": "fraud_detection",
    "rows_per_batch": 100,
    "batch_interval_seconds": 1.0,
    "timeout_minutes": 5,
})
print(resp.json())

In [ ]:
fraud_schema = StructType([
    StructField("transaction_id", StringType()),
    StructField("user_id", IntegerType()),
    StructField("merchant_id", IntegerType()),
    StructField("amount", DecimalType(10, 2)),
    StructField("currency", StringType()),
    StructField("merchant_category", StringType()),
    StructField("payment_method", StringType()),
    StructField("ip_address", StringType()),
    StructField("device_id", StringType()),
    StructField("latitude", DecimalType(8, 5)),
    StructField("longitude", DecimalType(9, 5)),
    StructField("card_type", StringType()),
    StructField("event_timestamp", StringType()),
])

fraud_df = read_kafka_stream("streaming-fraud-transactions", fraud_schema)

dbutils.fs.rm(f"{CHECKPOINT_BASE}/quickstart_fraud", recurse=True)
display(fraud_df, checkpointLocation=f"{CHECKPOINT_BASE}/quickstart_fraud")

---
## 3. Quick Neo4j Test

In [ ]:
driver = neo4j.GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

with driver.session() as session:
    result = session.run("RETURN 1 AS test")
    print(f"Neo4j connected: {result.single()['test']}")

    result = session.run("MATCH (n) RETURN labels(n)[0] AS label, count(n) AS count ORDER BY count DESC")
    records = [record.data() for record in result]
    if records:
        display(spark.createDataFrame(records))
    else:
        print("No nodes yet — run examples/neo4j_graph.ipynb to populate SLED data")

driver.close()

---
## 4. Quick PostgreSQL Test

In [ ]:
df = read_pg_table("kafka_event_logs")
print(f"Kafka event logs: {df.count()} rows")
display(df.limit(10))

---
## 5. Cleanup

In [ ]:
generators = requests.get(f"{API_URL}/kafka/generators", headers=HEADERS).json()
for g in generators:
    if g["status"] == "running":
        requests.post(f"{API_URL}/kafka/generators/{g['generator_id']}/stop", headers=HEADERS)
        print(f"Stopped {g['use_case']} generator {g['generator_id']}")

cleanup = requests.delete(f"{API_URL}/kafka/generators/cleanup", headers=HEADERS).json()
print(f"Cleaned up {cleanup.get('removed', 0)} generator(s)")